### Burgers' equation

$$
    \frac{\partial u(x, t)}{\partial t} + u(x, t) \, \frac{\partial u(x, t)}{\partial x} = \nu \, \frac{\partial ^ 2 u(x, t)}{\partial x ^ 2}
$$

We want to find its analytical solution given the initial shape $u_0(x) = u(x, 0)$

In [1]:
# Import SymPy
import sympy as smp

In [2]:
# Define symbols and functions
nu = smp.symbols('nu', real = True, constant = True, positive = True)
x, t = smp.symbols('x t', real = True, nonnegative = True)

u = smp.Function('u', real = True)(x, t)
phi = smp.Function('phi', real = True)(x, t)

In [3]:
# eq = 0
eq = smp.Derivative(u, t, 1) + u * smp.Derivative(u, x, 1) - nu * smp.Derivative(u, x, 2)
eq

-nu*Derivative(u(x, t), (x, 2)) + u(x, t)*Derivative(u(x, t), x) + Derivative(u(x, t), t)

The Cole–Hopf transformation suggests to write the solution as:

$$
    u(x, t) = - 2 \nu \, \frac{\partial \ln(\phi (x, t))}{\partial x}
$$

Let's do it

In [4]:
# Applying the Cole–Hopf transformation
eq = (((eq.subs(u, - 2 * nu * smp.Derivative(smp.ln(phi), x, 1))).doit().expand()) * phi ** 2 / (- 2 * nu)).expand()
eq

-nu*phi(x, t)*Derivative(phi(x, t), (x, 3)) + nu*Derivative(phi(x, t), x)*Derivative(phi(x, t), (x, 2)) + phi(x, t)*Derivative(phi(x, t), t, x) - Derivative(phi(x, t), t)*Derivative(phi(x, t), x)

This result:

$$
    - \nu \phi{\left(x,t \right)} \frac{\partial^{3}}{\partial x^{3}} \phi{\left(x,t \right)} + \nu \frac{\partial}{\partial x} \phi{\left(x,t \right)} \frac{\partial^{2}}{\partial x^{2}} \phi{\left(x,t \right)} + \phi{\left(x,t \right)} \frac{\partial^{2}}{\partial x\partial t} \phi{\left(x,t \right)} - \frac{\partial}{\partial t} \phi{\left(x,t \right)} \frac{\partial}{\partial x} \phi{\left(x,t \right)}
$$

Can be expressed as:

$$
    \phi ^ 2 (x, t) \, \frac{\partial}{\partial x} F(x, t)
$$

Where F is equal to:

$$
    F(x, t) = \frac{\frac{\partial \phi(x, t)}{\partial t} - \nu \, \frac{\partial ^ 2 \phi(x, t)}{\partial x ^ 2}}{\phi(x, t)}
$$

In [5]:
F = (smp.Derivative(phi, t, 1) - nu * smp.Derivative(phi, x, 2)) / phi # Computing F
dFdx = smp.Derivative(F, x, 1).doit().cancel() # Computing dF/dx
dFdx

(-nu*phi(x, t)*Derivative(phi(x, t), (x, 3)) + nu*Derivative(phi(x, t), x)*Derivative(phi(x, t), (x, 2)) + phi(x, t)*Derivative(phi(x, t), t, x) - Derivative(phi(x, t), t)*Derivative(phi(x, t), x))/phi(x, t)**2

In [6]:
smp.numer(dFdx) == eq

True

We can write:

$$
    \phi ^ 2 (x, t) \, \frac{\partial}{\partial x} \left( \frac{\frac{\partial \phi(x, t)}{\partial t} - \nu \, \frac{\partial ^ 2 \phi(x, t)}{\partial x ^ 2}}{\phi(x, t)} \right) = 0
$$

For $\phi(x, t) \neq 0$, we have to solve:

$$
    \frac{\partial \phi(x, t)}{\partial t} - \nu \, \frac{\partial ^ 2 \phi(x, t)}{\partial x ^ 2} = c(t) \, \phi(x, t)
$$

Where $c(t)$ is an arbitrary function. To eliminate it, let's define:

$$
    \theta(x, t) = \phi(x, t) e ^ {- \int _ 0 ^ t c(\tau) d \tau}
$$

This new function solves the diffusion equation:

In [7]:
tau = smp.symbols('tau', real = True, nonnegative = True) # tau
c = smp.Function('c', real = True)(t) # c function
theta = smp.Function('theta', real = True)(x, t) # New function

A = smp.Integral(c, t)
A

Integral(c(t), t)

In [8]:
# Write the equation to solve
eq = smp.Derivative(phi, t, 1) - nu * smp.Derivative(phi, x, 2) - c * phi
eq

-nu*Derivative(phi(x, t), (x, 2)) - c(t)*phi(x, t) + Derivative(phi(x, t), t)

In [9]:
# Substitute theta into it
(eq.subs(phi, theta * smp.exp(A))).doit().factor()

-(nu*Derivative(theta(x, t), (x, 2)) - Derivative(theta(x, t), t))*exp(Integral(c(t), t))

So, $\theta(x, t)$ solves:

$$
    \frac{\partial \theta(x, t)}{\partial t} = \nu \, \frac{\partial ^ 2 \theta(x, t)}{\partial x ^ 2}
$$

Let's write the intial condition:

In [10]:
# Initial functions
u0 = smp.Function('u_0', real = True)(x)
phi0 = smp.Function('phi_0', real = True)(x)
theta0 = smp.Function('theta_0', real = True)(x)

In [11]:
# Cole‑Hopf transformation of u0(x)
eq = smp.Eq(u0, - 2 * nu * smp.Derivative((smp.ln(phi0)), x, 1).doit())
eq

Eq(u_0(x), -2*nu*Derivative(phi_0(x), x)/phi_0(x))

In [12]:
# Cole‑Hopf transformation in terms of theta0(x)
eq = eq.subs(phi0, theta0 * smp.exp(A)).doit()
eq

Eq(u_0(x), -2*nu*Derivative(theta_0(x), x)/theta_0(x))

In [13]:
# Integrate both sides of the equation
eq = smp.Eq(smp.integrate(eq.rhs, x), smp.integrate(eq.lhs, x))
eq

Eq(-2*nu*log(theta_0(x)), Integral(u_0(x), x))

In [14]:
# Find theta0(x)
smp.solve(eq, theta0)[0]

exp(-Integral(u_0(x), x)/(2*nu))

Then, the initial condition for our diffusion equation is:

$$
    \theta _ 0 (x) = C_0 \, e^{- \frac{\int _ 0 ^ x u_{0}{\left(x' \right)}\, dx'}{2 \nu}}
$$

Where $C_0$ is an arbitrary constant.

The boundary conditions of the problem depends from the case. We'll study the infinite domain scenario, where $x \in [- \infty, + \infty]$. The diffusion equation has a fundumental solution. Choose $C_0 = 1$ for:

$$
    \theta(x,t) = \frac{1}{\sqrt{4\pi\nu t}} \int_{-\infty}^{\infty} e^{-\frac{(x-y)^2}{4\nu t}} \, \theta_0(y) \, dy
$$

This is the convolution with the Gaussian kernel (heat kernel). Now you can compute the solution $u(x, t)$

In [15]:
y = smp.symbols('y', real = True) # Dummy y
xi = smp.symbols('xi', real = True) # Dummy xi

# Computing the solutions
theta0_sol = smp.exp(- 1 / (2 * nu) * smp.Integral(u0.subs(x, xi), (xi, 0, x)))
theta_sol = 1 / smp.sqrt(4 * smp.pi * nu * t) * smp.Integral((theta0_sol.subs(x, y) * smp.exp(- ((x - y) ** 2) / (4 * nu * t))), (y, -smp.oo, smp.oo))
u_sol = (- 2 * nu * (smp.Derivative(theta_sol, x, 1)/ theta_sol))

In [16]:
theta0_sol

exp(-Integral(u_0(xi), (xi, 0, x))/(2*nu))

In [17]:
theta_sol = theta_sol.doit().simplify()
theta_sol

exp(-x**2/(4*nu*t))*Integral(exp(-Integral(u_0(xi), (xi, 0, y))/(2*nu))*exp(-y**2/(4*nu*t))*exp(x*y/(2*nu*t)), (y, -oo, oo))/(2*sqrt(pi)*sqrt(nu)*sqrt(t))

In [18]:
# Final solution (finally!)
u_sol = u_sol.doit().simplify()
u_sol

x/t - Integral(y*exp(-Integral(u_0(xi), (xi, 0, y))/(2*nu))*exp(-y**2/(4*nu*t))*exp(x*y/(2*nu*t)), (y, -oo, oo))/(t*Integral(exp(-Integral(u_0(xi), (xi, 0, y))/(2*nu))*exp(-y**2/(4*nu*t))*exp(x*y/(2*nu*t)), (y, -oo, oo)))